# 🌊 AquaYOLO: Underwater Marine Life Detection

This notebook provides an end-to-end training, evaluation, and diagnostic visualization pipeline for marine animal detection in Google Colab.

### 📌 Features in this Notebook:
- **Automated Multi-Dataset Harmonizer**: Specifically merges and remaps your 3 downloaded datasets without label conflict:
  1. `marine-life-pzpu1` (Roboflow YOLO format)
  2. `aquarium-data-cots` (Kaggle YOLO format)
  3. `sea-animals-image-dataste` (Kaggle classification images -> converted to detection targets)
- **High-Accuracy YOLO Training**: Resolves previous low-accuracy issues using transfer learning (`yolov8s.pt` / `FishInv.pt`), mosaic augmentation, and cosine LR decay.
- **All Metric Graphs Displayed in Output Cells**:
  - `results.png` (Training & Validation Losses, Precision, Recall, mAP@0.50, mAP@0.50:0.95)
  - `confusion_matrix.png` & `confusion_matrix_normalized.png`
  - `PR_curve.png` (Precision-Recall curve)
  - `F1_curve.png`, `P_curve.png`, `R_curve.png`
  - `val_batch0_pred.jpg` (Visual prediction comparisons)
- **Custom AquaYOLO Model**: Optional cell to train and test custom AquaYOLO (with SAC & TCG attention).

## 🚀 Step 1: Check GPU Runtime
Ensure your runtime is using a GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 📂 Step 2: Mount Google Drive
Mount Google Drive where your 3 downloaded dataset zip files are saved.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 3: Clone AquaYOLO Repository & Install Dependencies

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/Aravind8833/AquaYOLO.git
%cd /content/AquaYOLO

# Install dependencies
!pip install -q ultralytics opencv-python matplotlib seaborn pyyaml pandas pillow tqdm
print('\n Environment ready!')

## 📁 Step 4: Merge & Harmonize Your 3 Downloaded Zip Files

Enter the paths to your three zip files in Google Drive:
1. `marine-life-pzpu1.zip` (Roboflow)
2. `aquarium-data-cots.zip` (Kaggle)
3. `sea-animals-image-dataste.zip` (Kaggle)

The `dataset_merger.py` module will automatically:
- Extract each zip to local Colab SSD
- Harmonize conflicting class IDs into a unified marine taxonomy
- Convert classification images into detection targets
- Create `/content/dataset` with balanced `train`, `val`, and `test` splits and generate `data.yaml`.

In [ ]:
from pathlib import Path
from dataset_merger import merge_three_zips

# =====================================================================
# ⚙️ UPDATE THESE 3 PATHS TO YOUR ZIP FILES IN GOOGLE DRIVE:
# =====================================================================
ZIP_PATHS = [
    "/content/drive/MyDrive/marine-life-pzpu1.zip",        # Zip 1: Roboflow
    "/content/drive/MyDrive/aquarium-data-cots.zip",        # Zip 2: Kaggle COTS Aquarium
    "/content/drive/MyDrive/sea-animals-image-dataste.zip"  # Zip 3: Kaggle Sea Animals
]

# Or point to a single folder if you put all 3 zips in one folder:
DRIVE_ZIP_DIR = "/content/drive/MyDrive/marine_zips"
# =====================================================================

# Check zip paths
found_zips = [Path(p) for p in ZIP_PATHS if Path(p).is_file()]
if not found_zips and Path(DRIVE_ZIP_DIR).is_dir():
    found_zips = sorted(Path(DRIVE_ZIP_DIR).glob("*.zip"))

print(f"Found {len(found_zips)} zip files to process:")
for z in found_zips:
    print(f"  📦 {z.name} ({z.stat().st_size / (1024*1024):.1f} MB)")

if not found_zips:
    raise FileNotFoundError("⚠️ Please update ZIP_PATHS with the correct Google Drive locations of your zip files!")

# Run automated harmonizer
LOCAL_DATASET_DIR = Path("/content/dataset")
TEMP_EXTRACT_DIR = Path("/content/temp_raw_zips")

DATA_YAML_PATH = merge_three_zips(found_zips, LOCAL_DATASET_DIR, TEMP_EXTRACT_DIR)
print(f"\n Unified dataset ready at: {DATA_YAML_PATH}")

## 🏋️ Step 5: High-Accuracy Training (YOLO Benchmark)
Trains YOLOv8 with transfer learning, mosaic augmentation, and cosine learning rate decay.

In [ ]:
# Run high-accuracy training
!python train_yolo.py \
    --data "{DATA_YAML_PATH}" \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 16 \
    --imgsz 640 \
    --project /content/AquaYOLO/runs/yolo_train \
    --name colab_run

## 📊 Step 6: Display All Generated Metric Graphs
This displays every metric graph generated by YOLO directly in the notebook output:
1. **Training & Validation Loss / Metrics Curve** (`results.png`)
2. **Confusion Matrix & Normalized Confusion Matrix** (`confusion_matrix.png`)
3. **Precision-Recall Curve** (`PR_curve.png`)
4. **F1-Confidence Curve** (`F1_curve.png`)
5. **Precision & Recall Curves** (`P_curve.png`, `R_curve.png`)
6. **Validation Ground Truth vs Model Predictions** (`val_batch0_labels.jpg`, `val_batch0_pred.jpg`)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

RUN_DIR = Path("/content/AquaYOLO/runs/yolo_train/colab_run")

def show_graph(image_filename, title):
    p = RUN_DIR / image_filename
    if p.exists():
        print("=" * 75)
        print(f"{title}: {p.name}")
        print("=" * 75)
        display(Image(filename=str(p), width=900))
    else:
        print(f"{image_filename} not generated yet.")

# 1. Training & Validation Results (Losses, Precision, Recall, mAP50, mAP50-95)
show_graph("results.png", "TRAINING & VALIDATION METRICS OVER EPOCHS")

# 2. Precision-Recall Curve
show_graph("PR_curve.png", "PRECISION-RECALL (PR) CURVE")

# 3. F1-Confidence Curve
show_graph("F1_curve.png", "F1-SCORE VS CONFIDENCE CURVE")

# 4. Precision & Recall vs Confidence Curves
show_graph("P_curve.png", "PRECISION VS CONFIDENCE CURVE")
show_graph("R_curve.png", "RECALL VS CONFIDENCE CURVE")

# 5. Confusion Matrices
show_graph("confusion_matrix.png", "CONFUSION MATRIX")
show_graph("confusion_matrix_normalized.png", "NORMALIZED CONFUSION MATRIX")

# 6. Visual Detections on Validation Set (Ground Truth vs Model Prediction)
show_graph("val_batch0_labels.jpg", "GROUND TRUTH LABELS")
show_graph("val_batch0_pred.jpg", "MODEL PREDICTIONS ON VALIDATION BATCH")

## 📋 Step 7: View Numerical Metrics Table (Epoch by Epoch)

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print("\n--- FINAL EPOCHS SUMMARY ---")
    display(df.tail(10))
else:
    print("results.csv not found.")

## 🧠 Step 8: (Optional) Train Custom AquaYOLO Architecture
AquaYOLO model with CSPDarknet backbone, Spectral Attenuation Compensator (SAC), and Turbidity Contrast Gate (TCG).

In [ ]:
# Train custom AquaYOLO model
!python src/aquayolo/train.py \
    --data "{DATA_YAML_PATH.parent}" \
    --epochs 30 \
    --batch 4 \
    --size 640 \
    --lr 0.0001 \
    --out ./runs/aquayolo_custom_colab

# Show AquaYOLO loss curves
aqua_loss_plot = Path("./runs/aquayolo_custom_colab/loss_curve.png")
if aqua_loss_plot.exists():
    display(Image(filename=str(aqua_loss_plot), width=800))
aqua_comp_plot = Path("./runs/aquayolo_custom_colab/loss_components.png")
if aqua_comp_plot.exists():
    display(Image(filename=str(aqua_comp_plot), width=900))